In [ ]:
import sys

sys.path.append("../../src/")


import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from lightning import seed_everything

from model_evaluation.utils import compute_crps, compute_mpiw, compute_picp
from model_training.data_modules.utils import EPFDataModule

In [ ]:
seed_everything(0)

In [ ]:
n_runs = 10
standardization_case = "mean_std"

In [ ]:
# Load the data
val_date = "2022-12-01"
test_date = "2023-12-01"
end_date = "2024-11-30"
data_file_path = "../../data/processed/smard_data_201810010000_202501010000.npz"
data_module = EPFDataModule(
    data_file_path=data_file_path,
    val_date=val_date,
    test_date=test_date,
    end_date=end_date,
    batch_size=32,
    standardization_case=standardization_case,
)

train_input, train_labels = data_module.train_dataset[:]
val_input, val_labels = data_module.val_dataset[:]
test_input, test_labels = data_module.test_dataset[:]

data_input, data_labels = test_input, test_labels

data_labels = data_labels * data_module.scale_target + data_module.offset_target
data_labels = data_labels.detach().numpy().astype(np.float64)

In [ ]:
quantiles = np.linspace(0.01, 0.99, 99)
confidence_levels = np.flip(
    np.array([quantiles[-i - 1] - quantiles[i] for i in range(len(quantiles) // 2)])
)

In [ ]:
models_dict = []

In [ ]:
model_name = "XGBoost_CP"
file_path = f"../evaluate_models/results/archive/{model_name}.pkl"
with open(
    file_path,
    "rb",
) as f:
    models_dict.extend(pickle.load(f))

In [ ]:
models_dict[0].keys()

In [ ]:
models_dict[0]["model_name"]

In [ ]:
models_dict[0]["prediction"].shape

In [ ]:
models_dict[0]["quantile"].shape

In [ ]:
# compute metrics
for model in models_dict:
    # compute mae
    model["mae"] = np.mean(np.abs(model["prediction"] - data_labels), axis=(1, 2))
    model["mae_mean"] = np.mean(model["mae"])
    model["mae_std"] = np.std(model["mae"])

    # compute rmse
    model["rmse"] = np.sqrt(
        np.mean((model["prediction"] - data_labels) ** 2, axis=(1, 2))
    )
    model["rmse_mean"] = np.mean(model["rmse"])
    model["rmse_std"] = np.std(model["rmse"])

    # compute crps score (approximated with average pinball score)
    model["crps"] = compute_crps(quantiles, model["quantile"], data_labels)
    model["crps_mean"] = np.mean(model["crps"])
    model["crps_std"] = np.std(model["crps"])

    # compute prediction interval coverage probability (PICP)
    model["coverage"] = compute_picp(quantiles, model["quantile"], data_labels)
    model["coverage_mean"] = np.mean(model["coverage"], axis=0)
    model["coverage_std"] = np.std(model["coverage"], axis=0)

    # compute mean absolute average coverage error (MAACE)
    model["maace"] = np.mean(np.abs(model["coverage"] - confidence_levels), axis=1)
    model["maace_mean"] = np.mean(model["maace"])
    model["maace_std"] = np.std(model["maace"])

    # compute mean prediction interval width (MPIW)
    model["mpiw"] = compute_mpiw(quantiles, model["quantile"])
    model["mpiw_mean"] = np.mean(model["mpiw"], axis=0)
    model["mpiw_std"] = np.std(model["mpiw"], axis=0)

In [ ]:
metrics = (
    ["mae", "mae_std", "rmse", "rmse_std", "crps", "crps_std", "maace", "maace_std"]
    if n_runs > 1
    else ["mae", "rmse", "crps", "maace"]
)
results = {
    metric: [np.mean(model[metric]) for model in models_dict] for metric in metrics
}

model_names = [model["model_name"] for model in models_dict]
results_df = pd.DataFrame(results, index=model_names)
results_df

In [ ]:
# plot coverage probability
plt.figure(figsize=(12, 8))
i = 0
for model in models_dict:
    plt.plot(
        confidence_levels * 100,
        model["coverage_mean"] * 100,
        label=model["model_name"],
        alpha=0.8,
    )

plt.plot(confidence_levels * 100, confidence_levels * 100, linestyle="--")
plt.xlim(0, 100)
plt.grid(alpha=0.4)
plt.xlabel("Confidence level (%)")
plt.ylabel("Coverage (%)")
plt.legend()
plt.show()

In [ ]:
# plot mean prediction interval width
plt.figure(figsize=(12, 8))
for model in models_dict:
    plt.plot(
        confidence_levels * 100,
        model["mpiw_mean"],
        label=model["model_name"],
        # marker="x",
        # markersize=4,
        alpha=0.8,
    )

plt.xlim(0, 100)
# plt.ylim(0, 100)
plt.grid(alpha=0.4)
plt.xlabel("Significance level (%)")
plt.ylabel("Mean prediction interval width")
plt.legend()
plt.show()

In [ ]:
# plot mean prediction interval width against coverage probability
plt.figure(figsize=(12, 8))
for model in models_dict:
    plt.plot(
        model["coverage_mean"] * 100,
        model["mpiw_mean"],
        label=model["model_name"],
        marker="x",
        markersize=4,
        alpha=0.8,
    )
plt.xlim(0, 100)
plt.grid(alpha=0.4)
plt.xlabel("Coverage (%)")
plt.ylabel("Mean prediction interval width")
plt.legend()
plt.show()

In [ ]:
# save results
import os

if not os.path.exists("results/metric_evaluation/" + model_name + ".pkl"):
    with open("results/metric_evaluation/" + model_name + ".pkl", "wb") as f:
        pickle.dump(models_dict, f)
